# Data Profiling: NYC Yellow Taxi 2024

This notebook profiles the cleaned taxi trip dataset for 2024. Before running any analysis or building dashboards, I want to make sure I actually understand what the data looks like — the shape of distributions, where the outliers are, whether the cleaning step worked, and which columns are actually useful.

The source table is `trips_clean`, which was created in `02_clean.sql` by filtering out trips with invalid timestamps, zero fares, and impossible distances from the raw ingested data.

This is not analysis yet. This is just getting familiar with the data before drawing any conclusions from it.

## 0. Setup & Database Connection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DB_PASSWORD = 'user'
DB_USER     = 'postgres'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'NYC Yellow Taxi Trip Report Data 2024'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

plt.rcParams['figure.figsize']  = (12, 5)
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False
plt.rcParams['font.family']        = 'DejaVu Sans'
PALETTE = sns.color_palette('Blues_d', 8)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('Connected to PostgreSQL:', result.scalar()[:40])

## 1. Dataset Overview & Shape

Before profiling anything, we need to understand the basic shape of the data like how many rows, how many columns, and the types of data as well as the time range.

In [ ]:
overview = pd.read_sql_query('''
    SELECT
        COUNT(*)                                                  AS total_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NULL)           AS clean_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)       AS flagged_rows,
        ROUND(COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)
              * 100.0 / COUNT(*), 2)                             AS pct_flagged,
        MIN(tpep_pickup_datetime)                                 AS earliest,
        MAX(tpep_pickup_datetime)                                 AS latest,
        COUNT(DISTINCT DATE_TRUNC(\'month\', tpep_pickup_datetime)) AS months_present
    FROM trips_clean
''', con=engine)

print('=== DATASET OVERVIEW ===')
print(f'Total Rows   : {overview["total_rows"][0]:>15,.0f}')
print(f'Clean Rows   : {overview["clean_rows"][0]:>15,.0f}')
print(f'Flagged Rows : {overview["flagged_rows"][0]:>15,.0f}  ({overview["pct_flagged"][0]}%)')
print(f'Date Range   : {overview["earliest"][0]} → {overview["latest"][0]}')
print(f'Months       : {overview["months_present"][0]} (full year 2024)')

In [ ]:
# Columns schema
schema = pd.read_sql_query('''
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_name = \'trips_clean\'
    ORDER BY ordinal_position
''', con=engine)

print(f'\n=== SCHEMA: {len(schema)} COLUMNS ===')
display(schema)

In [ ]:
# Load a sample for in-memory profiling to avoid loading 39M rows everytime
# Use TABLESAMPLE for a random sample
sample = pd.read_sql_query('''
    SELECT *
    FROM trips_clean
    TABLESAMPLE BERNOULLI(0.5)   -- ~0.5%% of rows, roughly 200k rows
    LIMIT 200000
''', con=engine)

print(f'Sample loaded: {len(sample):,} rows x {len(sample.columns)} columns')
sample.head(3)

## 2. Missing Value Analysis

Missing values can silently distort analysis. We check every column for nulls — both in terms of raw count and percentage — to decide if any columns need special handling before we proceed.

In [ ]:
null_counts = sample.isnull().sum().reset_index()
null_counts.columns = ['column', 'null_count']
null_counts['pct_null'] = (null_counts['null_count'] / len(sample) * 100).round(2)
null_counts = null_counts[null_counts['null_count'] > 0].sort_values('pct_null', ascending=False)

print('=== MISSING VALUE REPORT ===')
if len(null_counts) == 0:
    print('No missing values found in any column.')
else:
    display(null_counts)

In [ ]:
# Visualise missing values if any exist
if len(null_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(null_counts) * 0.5)))
    bars = ax.barh(null_counts['column'], null_counts['pct_null'], color=PALETTE[2])
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    for bar, pct in zip(bars, null_counts['pct_null']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('C:/Users/matth/GitTaxiProject/reports/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing value chart needed — all columns are complete.')

## 3. Numeric Column Distributions

We look at the spread of numeric columns using descriptive statistics (mean, median, std, min/max, percentiles). This tells us if values are concentrated in a narrow range or spread out widely — which has big implications for fare modelling and demand forecasting.

In [ ]:
# Defining numeric columns
numeric_cols = [
    'trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
    'trip_duration_min', 'avg_speed_mph', 'passenger_count',
    'congestion_surcharge', 'tolls_amount', 'extra'
]

# Using only clean rows
clean = sample[sample['suspicious_flag'].isna()].copy()

# Descriptive statistics
desc = clean[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
desc = desc.round(2)
print('=== DESCRIPTIVE STATISTICS (CLEAN ROWS ONLY) ===')
display(desc)

In [ ]:
# Distribution histograms for key fare and trip metrics
plot_cols = [
    ('fare_amount',       'Fare Amount ($)',         50, (0, 80)),
    ('tip_amount',        'Tip Amount ($)',           40, (0, 30)),
    ('trip_distance',     'Trip Distance (miles)',    50, (0, 30)),
    ('trip_duration_min', 'Trip Duration (minutes)',  50, (0, 90)),
    ('avg_speed_mph',     'Average Speed (mph)',      50, (0, 60)),
    ('total_amount',      'Total Amount ($)',         50, (0, 100)),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, label, bins, xlim) in zip(axes, plot_cols):
    data = clean[col].dropna()
    data = data[(data >= xlim[0]) & (data <= xlim[1])]
    ax.hist(data, bins=bins, color=PALETTE[3], edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {data.median():.2f}')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.suptitle('NYC Yellow Taxi 2024 — Key Metric Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/distributions.png')

## 4. Outlier Detection

We use two standard techniques to identify extreme values:

- **IQR method**: Flags values below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`. Good for skewed distributions.
- **Z-Score method**: Flags values more than 3 standard deviations from the mean. Better for normal-ish distributions.

These are NOT automatically removed — we document them and decide case by case.

In [ ]:
def iqr_outliers(series):
    """Returns count and % of outliers using the IQR method."""
    s = series.dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = s[(s < lower) | (s > upper)]
    return len(outliers), round(len(outliers) / len(s) * 100, 2), round(lower, 2), round(upper, 2)

def zscore_outliers(series, threshold=3):
    """Returns count and % of outliers using Z-Score method."""
    s = series.dropna()
    z = np.abs(stats.zscore(s))
    outliers = z[z > threshold]
    return len(outliers), round(len(outliers) / len(s) * 100, 2)

outlier_cols = ['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min', 'avg_speed_mph', 'total_amount']

results = []
for col in outlier_cols:
    iqr_n, iqr_pct, lower, upper = iqr_outliers(clean[col])
    z_n, z_pct = zscore_outliers(clean[col])
    results.append({
        'column': col,
        'iqr_outliers': iqr_n,
        'iqr_pct': iqr_pct,
        'iqr_lower_bound': lower,
        'iqr_upper_bound': upper,
        'zscore_outliers': z_n,
        'zscore_pct': z_pct
    })

outlier_df = pd.DataFrame(results)
print('=== OUTLIER DETECTION REPORT ===')
display(outlier_df)

In [ ]:
# Boxplots to visaulise outliers
box_cols = ['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, box_cols):
    data = clean[col].dropna()
    cap = data.quantile(0.99)
    ax.boxplot(data[data <= cap], vert=True, patch_artist=True,
               boxprops=dict(facecolor=PALETTE[2], color=PALETTE[5]),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='.', markerfacecolor=PALETTE[1], markersize=1, alpha=0.3))
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xticks([])

plt.suptitle('Boxplots — Key Metrics (capped at 99th percentile)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/boxplots.png')

In [ ]:
pct_levels = [0.50, 0.90, 0.95, 0.99, 0.999]
pct_table = clean[['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min', 'avg_speed_mph']]\
    .quantile(pct_levels).round(2)
pct_table.index = [f'{int(p*100)}th percentile' for p in pct_levels]

print('=== PERCENTILE TABLE ===')
display(pct_table)